# AML Trainer

Treino offline do classificador (Random Forest, Spark MLlib).
Lê `LI-Small_Trans.csv` de HDFS, faz feature engineering, treina e persiste o `PipelineModel` em HDFS.
O `PipelineModel` é depois carregado pelos consumers (Kafka e file-based) e aplicado em streaming.

In [2]:
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType,
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("AML Model Trainer") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [8]:
HDFS_BASE = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
TRAIN_PATH = f"{HDFS_BASE}/dataset/LI-Small_Trans.csv"  #Onde o dataset de treinamento está guardado
MODEL_PATH = f"{HDFS_BASE}/model/rf_aml_pipeline"       #Onde o modelo treinado (Random Forest para AML*) está guardado

aml_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("From Account", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("To Account", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True),
])

In [9]:
df = spark.read \
    .option("header", True) \
    .option("timestampFormat", "yyyy/MM/dd HH:mm") \
    .schema(aml_schema) \
    .csv(TRAIN_PATH)

total = df.count()
positives = df.filter("`Is Laundering` = 1").count()
print(f"Linhas: {total:,}  Laundering: {positives:,} ({positives/total:.4%})")

Linhas: 6,924,049  Laundering: 3,565 (0.0515%)


In [10]:
df.show(5)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+
|          Timestamp|From Bank|From Account|To Bank|To Account|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-------------+
|2022-09-01 00:08:00|       11|   8000ECA90|     11| 8000ECA90|      3195403.0|         US Dollar|  3195403.0|       US Dollar|  Reinvestment|            0|
|2022-09-01 00:21:00|     3402|   80021DAD0|   3402| 80021DAD0|        1858.96|         US Dollar|    1858.96|       US Dollar|  Reinvestment|            0|
|2022-09-01 00:00:00|       11|   8000ECA90|   1120| 8006AA910|       592571.0|         US Dollar|   592571.0|       US Dollar|        Cheque|            0|
|2022-09-01 00:16:00|     3814|   8006AD080|   3814| 8006A

## Feature engineering

Idêntica nos consumers — para evitar duplicação, está empacotada numa função que será reproduzida nos notebooks dos consumers.

In [11]:
def add_features(d):
    return (
        d.withColumn("log_amt_paid", F.log1p(F.col("Amount Paid")))
         .withColumn("log_amt_recv", F.log1p(F.col("Amount Received")))
         .withColumn("amt_diff", F.abs(F.col("Amount Paid") - F.col("Amount Received")))
         .withColumn("same_bank", (F.col("From Bank") == F.col("To Bank")).cast("int"))
         .withColumn("ccy_mismatch", (F.col("Receiving Currency") != F.col("Payment Currency")).cast("int"))
         .withColumn("hour", F.hour("Timestamp"))
    )

featured = add_features(df) \
    .withColumnRenamed("Is Laundering", "label") \
    .na.fill({"hour": 0})

train, test = featured.randomSplit([0.8, 0.2], seed=42)
train.cache()
print(f"Train: {train.count():,}  Test: {test.count():,}")

Train: 5,539,402  Test: 1,384,647


In [13]:
# Pesos de classe para o forte desbalanceamento (laundering << 1%)
pos = train.filter("label = 1").count()
neg = train.filter("label = 0").count()
w_pos = neg / (pos + neg)
w_neg = pos / (pos + neg)
print(f"weight pos={w_pos:.4f}  neg={w_neg:.4f}")

train_w = train.withColumn(
    "class_weight",
    F.when(F.col("label") == 1, F.lit(w_pos)).otherwise(F.lit(w_neg)),
)

weight pos=0.9995  neg=0.0005


In [15]:
train.show(2)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-----+-----------------+-----------------+--------+---------+------------+----+
|          Timestamp|From Bank|From Account|To Bank|To Account|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|label|     log_amt_paid|     log_amt_recv|amt_diff|same_bank|ccy_mismatch|hour|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-----+-----------------+-----------------+--------+---------+------------+----+
|2022-09-01 00:00:00|        0|   800062F80|      3| 800064730|        3857.49|              Euro|    3857.49|            Euro|          Cash|    0|8.258031194213594|8.258031194213594|     0.0|        0|           0|   0|
|2022-09-01 00:00:00|        0|   8002139E0|      0| 8002139E0|          19.22|              Euro|      19.22|  

In [16]:
train_w.show(2)

+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-----+-----------------+-----------------+--------+---------+------------+----+--------------------+
|          Timestamp|From Bank|From Account|To Bank|To Account|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|label|     log_amt_paid|     log_amt_recv|amt_diff|same_bank|ccy_mismatch|hour|        class_weight|
+-------------------+---------+------------+-------+----------+---------------+------------------+-----------+----------------+--------------+-----+-----------------+-----------------+--------+---------+------------+----+--------------------+
|2022-09-01 00:00:00|        0|   800062F80|      3| 800064730|        3857.49|              Euro|    3857.49|            Euro|          Cash|    0|8.258031194213594|8.258031194213594|     0.0|        0|           0|   0|5.143154441580517E-4|
|2022-09-01 00:00:00|       

In [17]:
cat_cols = ["Receiving Currency", "Payment Currency", "Payment Format"]
num_cols = ["log_amt_paid", "log_amt_recv", "amt_diff", "same_bank", "ccy_mismatch", "hour", "From Bank", "To Bank"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe", handleInvalid="keep") for c in cat_cols]

assembler = VectorAssembler(
    inputCols=num_cols + [f"{c}_ohe" for c in cat_cols],
    outputCol="features",
    handleInvalid="keep",
)

rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="class_weight",
    numTrees=50,
    maxDepth=10,
    seed=42,
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [18]:
t0 = time.time()
model = pipeline.fit(train_w)
train_time = time.time() - t0
print(f"Tempo de treino: {train_time:.1f}s")

Tempo de treino: 142.8s


In [19]:
preds = model.transform(test)

auc = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC",
).evaluate(preds)
f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1",
).evaluate(preds)

print(f"AUC: {auc:.4f}   F1: {f1:.4f}\n")
print("Matriz de confusão:")
preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

AUC: 0.9443   F1: 0.9548

Matriz de confusão:
+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0|1265551|
|    0|       1.0| 118380|
|    1|       0.0|    189|
|    1|       1.0|    527|
+-----+----------+-------+



In [20]:
model.write().overwrite().save(MODEL_PATH)
print(f"Modelo guardado em {MODEL_PATH}")

Modelo guardado em hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/model/rf_aml_pipeline
